# Python results — complete run

Finishes every Python number for the paper: the Coder-1.5B wrap-up
(frozen-split baselines, selectivity, accuracy deltas), the multi-model
sweep, and a consolidated results table.

**Runtime → GPU (L4)**, then **Runtime → Run all**. Every step is
resumable — if the session dies, Run all again and it continues.

Edit `MODELS` in cell 3 to add/remove sweep models.

In [ ]:
# 1 — setup: clone/pull, deps, token, restore prior work from Drive
# Reproduction/audit runs: set PIN_COMMIT to a full SHA to run vetted code
# instead of the moving branch (the branch is the default for active work
# on this private repo). Use a READ-only HF token in Colab secrets.
PIN_COMMIT = ""  # e.g. "d7ac7a9..."
import os, pathlib
if not pathlib.Path("/content/mech-interp-coding-llms").exists():
    !git clone -q -b nolan-boolean https://github.com/nolanlwin/mech-interp-coding-llms.git /content/mech-interp-coding-llms
%cd /content/mech-interp-coding-llms
!git fetch -q origin
_old = !git rev-parse HEAD
if PIN_COMMIT:
    !git checkout -q {PIN_COMMIT}
else:
    !git pull -q
_new = !git rev-parse HEAD
if _old[0] != _new[0]:
    print("=" * 70)
    print("CODE CHANGED since this runtime last ran — review before trusting")
    print("the run with your HF token / Drive. New commits:")
    !git log --oneline {_old[0]}..{_new[0]}
    print("=" * 70)
!git log --oneline -1
!pip install -q transformers==5.8.0 tree_sitter "tree-sitter-java>=0.23.5" "tree-sitter-go>=0.25.0" \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" "tree-sitter-ruby>=0.23.1" \
  scikit-learn scipy huggingface_hub
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/xlcost"
!mkdir -p outputs/activations_xlcost outputs/probe_results outputs/xlcost_occ data/xlcost outputs/xlcost_occ_renamed data/xlcost_renamed
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n {DEST}/probe_results/* outputs/probe_results/ 2>/dev/null || true
!cp -n {DEST}/xlcost_occ/* outputs/xlcost_occ/ 2>/dev/null || true
!cp -n {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true
!cp -rn {DEST}/xlcost_occ_renamed/* outputs/xlcost_occ_renamed/ 2>/dev/null || true
!cp -rn {DEST}/data_xlcost_renamed/* data/xlcost_renamed/ 2>/dev/null || true
print("setup complete")

In [ ]:
# 2 — wrap up Python x Qwen2.5-Coder-1.5B: baselines on the frozen split,
#     selectivity, and per-condition accuracy deltas. CPU, ~2 min.
!python scripts/baselines.py run \
  --occurrences outputs/xlcost_occ/python_train.jsonl \
  --canonical data/xlcost/python_train.jsonl \
  --sample-ids outputs/probe_results/python_train_qwen25coder15b_problem.json.sample_ids.json \
  --split-policy repo \
  --output outputs/probe_results/python_train_qwen25coder15b_baselines_capped.json
import json
import numpy as np
c0 = json.load(open("outputs/probe_results/python_train_qwen25coder15b_problem.json"))
print("C0 selectivity:", round(c0.get("selectivity_macro_f1", float("nan")), 4),
      "| control-task F1:", round(c0["control_task"]["test_macro_f1_mean"], 4))
p0 = {p["occurrence_id"]: p for p in c0["test_predictions"]}
for c in ["C1", "C2", "C3", "C4", "C5"]:
    try:
        r = json.load(open(f"outputs/probe_results/python_train_{c}_qwen25coder15b_problem.json"))
    except FileNotFoundError:
        print(f"{c}: missing — re-run run_renaming.sh for Coder-1.5B")
        continue
    pc = {p["occurrence_id"]: p for p in r["test_predictions"]}
    sh = sorted(set(p0) & set(pc))
    a0 = np.mean([p0[k]["y_pred"] == p0[k]["y_true"] for k in sh])
    ac = np.mean([pc[k]["y_pred"] == pc[k]["y_true"] for k in sh])
    nd = sum(pc[k]["y_pred"] != p0[k]["y_pred"] for k in sh)
    print(f"{c}: acc delta {ac - a0:+.4f}   predictions changed on {nd}/{len(sh)}")

In [ ]:
# 3 — model sweep: full pass + renaming experiment per model. GPU.
#     ~45 min for the 1.5B, ~90 min for the 7B. Resumable per step.
MODELS = [
    "Qwen/Qwen2.5-1.5B",      # base-vs-Coder contrast at identical scale
    "bigcode/starcoder2-7b",  # different family, tokenizer, pretraining corpus
]
for m in MODELS:
    print(f"\n############ {m} ############")
    !bash scripts/run_language.sh Python {m} train
    !bash scripts/run_renaming.sh Python {m} train
    print(f"checkpoint: saving {m} to Drive")
    !mkdir -p {DEST}/probe_results {DEST}/stores
    !cp -r outputs/probe_results/* {DEST}/probe_results/ 2>/dev/null || true
    !cp -r outputs/activations_xlcost/* {DEST}/stores/ 2>/dev/null || true

In [ ]:
# 4 — consolidated Python results table (reads whatever exists)
import glob
import json
import re
probes = sorted(glob.glob("outputs/probe_results/python_train_*_problem.json"))
models = sorted({m.group(1) for f in probes
                 if (m := re.match(r".*python_train_(?!C\d)(\w+)_problem\.json$", f))})
print(f"{'model':<20}{'C0 F1':>8}{'select.':>9}{'best base':>11}"
      + "".join(f"{c:>9}" for c in ["dC1", "dC2", "dC3", "dC4", "dC5"]))
for m in models:
    try:
        c0 = json.load(open(f"outputs/probe_results/python_train_{m}_problem.json"))
    except FileNotFoundError:
        continue
    f1 = c0["aggregate"]["test_macro_f1_mean"]
    sel = c0.get("selectivity_macro_f1", float("nan"))
    try:
        b = json.load(open(f"outputs/probe_results/python_train_{m}_baselines_capped.json"))
        best = b["strongest_baseline_macro_f1"]
    except FileNotFoundError:
        best = float("nan")
    row = f"{m:<20}{f1:>8.4f}{sel:>9.4f}{best:>11.4f}"
    for c in ["C1", "C2", "C3", "C4", "C5"]:
        try:
            d = json.load(open(f"outputs/probe_results/python_train_{c}_{m}_delta_vs_C0.json"))
            row += f"{d['delta']:>+9.4f}"
        except FileNotFoundError:
            row += f"{'—':>9}"
    print(row)
print("\ndeltas = condition minus C0, paired on shared test occurrences (full CIs in the *_delta_vs_C0.json files)")

In [ ]:
# 5 — save everything to Drive (run after every milestone)
!mkdir -p {DEST}/probe_results {DEST}/stores {DEST}/xlcost_occ {DEST}/data_xlcost {DEST}/xlcost_occ_renamed {DEST}/data_xlcost_renamed
!cp -r outputs/probe_results/* {DEST}/probe_results/ 2>/dev/null || true
!cp -r outputs/xlcost_occ/* {DEST}/xlcost_occ/ 2>/dev/null || true
!cp -r data/xlcost/* {DEST}/data_xlcost/ 2>/dev/null || true
!cp -r outputs/xlcost_occ_renamed/* {DEST}/xlcost_occ_renamed/ 2>/dev/null || true
!cp -r data/xlcost_renamed/* {DEST}/data_xlcost_renamed/ 2>/dev/null || true
!cp -r outputs/activations_xlcost/* {DEST}/stores/
!ls {DEST}/probe_results/ | tail -5
print("saved")